In [1]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
##S3_BUCKET = "s3://ads508-housing-data-faye"

S3_BUCKET = "sagemaker-studio-mru71pwhjb"
ID_COLS = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

paths = {
    "zhvi": f"s3://{S3_BUCKET}/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv",
    "inv": f"s3://{S3_BUCKET}/Metro_invt_fs_uc_sfrcondo_sm_month.csv",
    "temp": f"s3://{S3_BUCKET}/Metro_market_temp_index_uc_sfrcondo_month.csv",
    "sale": f"s3://{S3_BUCKET}/Metro_median_sale_price_uc_sfrcondo_sm_week.csv"
}

In [ ]:
# 2. DATA LOADING & CLEANING
def load_and_clean(path, val_name, is_weekly=False):
    df = pd.read_csv(path).drop_duplicates()
    if "RegionType" in df.columns:
        df = df[df["RegionType"] == "msa"]
    df = df[df["RegionName"] != "United States"]
    
    # Wide to Long
    date_cols = [c for c in df.columns if c not in ID_COLS]
    df_long = df.melt(id_vars=ID_COLS, value_vars=date_cols, var_name="Date", value_name=val_name)
    df_long["Date"] = pd.to_datetime(df_long["Date"], errors="coerce")
    
    if is_weekly:
        df_long["Date"] = df_long["Date"].dt.to_period("M").dt.to_timestamp("M")
        return df_long.groupby(["RegionID", "RegionName", "StateName", "Date"], as_index=False)[val_name].mean()
    
    return df_long.dropna(subset=["Date"])

# Process datasets
zhvi = load_and_clean(paths['zhvi'], "ZHVI")
inv = load_and_clean(paths['inv'], "Inventory")
temp = load_and_clean(paths['temp'], "MarketTemp")
sale = load_and_clean(paths['sale'], "MedianSalePrice", is_weekly=True)

In [ ]:
# 3. MERGING & FEATURE ENGINEERING
merge_keys = ["RegionID", "RegionName", "StateName", "Date"]
df = zhvi.merge(inv[merge_keys + ["Inventory"]], on=merge_keys, how="left") \
         .merge(temp[merge_keys + ["MarketTemp"]], on=merge_keys, how="left") \
         .merge(sale, on=merge_keys, how="left")

# Sort and generate time-series features
df = df.dropna(subset=["ZHVI"]).sort_values(["RegionName", "Date"])
df["Year"], df["Month"] = df["Date"].dt.year, df["Date"].dt.month

# Lags and Percent Changes
for col in ["ZHVI", "Inventory", "MarketTemp"]:
    df[f"{col}_Lag1"] = df.groupby("RegionName")[col].shift(1)
df["ZHVI_PctChange"] = df.groupby("RegionName")["ZHVI"].pct_change()
df["Inventory_PctChange"] = df.groupby("RegionName")["Inventory"].pct_change()

# Final data selection (removing rows with no lag data)
df = df.dropna(subset=["ZHVI_Lag1", "Inventory_Lag1", "MarketTemp_Lag1"]).copy()

In [ ]:
# 4. TRAIN/TEST SPLIT
feature_cols = ["StateName", "Year", "Month", "Inventory", "MarketTemp", "MedianSalePrice", 
                "ZHVI_Lag1", "Inventory_Lag1", "MarketTemp_Lag1", "ZHVI_PctChange", "Inventory_PctChange"]
target_col = "ZHVI"

cutoff_date = df["Date"].quantile(0.80)
train_df = df[df["Date"] <= cutoff_date]
test_df = df[df["Date"] > cutoff_date]

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

In [ ]:
# 5. MODELING PIPELINE
numeric_features = [c for c in feature_cols if c != "StateName"]
categorical_features = ["StateName"]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), 
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1))
])

model.fit(X_train, y_train)
preds = model.predict(X_test)

In [ ]:
# 6. EVALUATION
print(f"MAE: {mean_absolute_error(y_test, preds):.2f}")
print(f"RMSE: {math.sqrt(mean_squared_error(y_test, preds)):.2f}")
print(f"R2 Score: {r2_score(y_test, preds):.4f}")

In [ ]:
# 7. EXPORT
df.to_csv(f"s3://{S3_BUCKET}/prepared/full_prepared_dataset.csv", index=False)